# Many sandboxes in parallel

This notebook creates 20 sandboxes at once, runs a test in each and deletes them again, with `AsyncSandboxClient`. A preset's sandboxes start from its snapshot, so each one is created and started in well under a second. See the [README](./README.md) for how many sandboxes can run at the same time.

Sandboxes must be enabled on your prokube cluster, and the Lab needs sandbox access.

## Setup

In [ ]:
%pip install -q "prokube-sdk @ git+https://github.com/prokube/prokube-sdk@sandboxv3/main"

## The test

Each sandbox checks one candidate implementation of a function, as in an eval or a test matrix. Every fourth candidate has a bug:

In [ ]:
N = 20
PRESET = "pk-default-small"

CANDIDATES = [f"def add(a, b):\n    return a + b{' + 1' if i % 4 == 3 else ''}\n" for i in range(N)]
TEST = "assert add(2, 3) == 5, f'add(2, 3) returned {add(2, 3)}'\n"

A new sandbox is suspended and takes no room, so all of them are created at once. Running sandboxes need CPU and memory in the worker pool, so at most `RUNNING` of them run at the same time:

In [ ]:
import asyncio

RUNNING = 4
room = asyncio.Semaphore(RUNNING)


async def evaluate(sandboxes, i):
    name = f"parallel-{i}"
    await sandboxes.create(name, preset=PRESET)
    async with room:
        try:
            await sandboxes.resume(name)
            result = await sandboxes.run(name, "python3 -", stdin=CANDIDATES[i] + TEST)
        finally:
            await sandboxes.delete(name)
    return result.exit_code == 0

## Run

In a Lab with sandbox access, `AsyncSandboxClient()` needs no configuration: it uses the Lab's workspace and identity.

In [ ]:
import time

from prokube.sandbox import AsyncSandboxClient

async with AsyncSandboxClient() as sandboxes:
    start = time.perf_counter()
    passed = await asyncio.gather(*(evaluate(sandboxes, i) for i in range(N)))
print(f"{N} sandboxes created, used and deleted in {time.perf_counter() - start:.1f} s")
print("failed:", [i for i, ok in enumerate(passed) if not ok])